<a href="https://colab.research.google.com/github/zainabbio/Data_Science/blob/main/New_MR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install core MR packages
install.packages("remotes")
remotes::install_github("MRCIEU/TwoSampleMR")
devtools::install_github("mrcieu/ieugwasr")
install.packages("data.table")
install.packages("coloc")

In [ ]:
## MR analysis testing causal effects of GLP-1R effect on outcomes selected
rm(list=ls())
gc()
library(TwoSampleMR)
library(coloc)
library(vroom)


## Step 1:  Constructing exposure data with instruments validation
data<-read.csv("multi-omic_qtl_instruments.csv")
#exposure data
data <- data.frame(SNP = data$SNP,
                   chr = data$chr,
                   pos = data$pos,
                   beta = as.numeric(data$beta),
                   se = as.numeric(data$se),
                   effect_allele = data$effect_allele,
                   other_allele = data$other_allele,
                   eaf=data$effect_allele_freq,
                   pval = as.numeric(data$pval),
                   Phenotype = data$Phenotype,
                   samplesize = data$samplesize,
                   info = paste(data$Study,data$cis_trans,sep="_"))
data <- format_data(data, type="exposure", phenotype_col = "Phenotype",chr_col = "chr",pos_col = "pos",samplesize_col = "samplesize",info_col = "info")
#ld clumping
data <-clump_data(data,clump_kb=10000,clump_r2=0.001,clump_p1=1,clump_p2=1,pop="EUR")
# F statistics>10
# Calculate F-statistics using R2 and MAF
exposure_dat <- cbind(data, fstatistics = 1)
for (s in 1:nrow(exposure_dat)){
  beta <- exposure_dat[s, "beta.exposure"]
  n <- exposure_dat[s, "samplesize.exposure"]
  eaf <- exposure_dat[s, "eaf.exposure"]

  # Calculate R2 using the formula: R2 = beta^2 * 2 * MAF * (1-MAF)
  r2 <- beta^2 * 2 * eaf * (1 - eaf)

  # Calculate F-statistic
  exposure_dat[s, "fstatistics"] <- (r2 * (n - 2)) / (1 - r2)
}

print(min(exposure_dat$fstatistics))
print(max(exposure_dat$fstatistics))
exposure_dat <- exposure_dat[exposure_dat$fstatistics > 10, ]


## Step 2:  MR analysis testing causal effects of GLP-1R expression on positive control/overall ovarian cancer/EOC subtypes
#outcome data
outcomelist<-read.csv("outcomelist.csv")
i<-1
out_data<- vroom(outcomelist[i,"filename"])
outcome_dat<-format_data(
    out_data,
    type = "outcome",
    snp_col = "SNP",
    beta_col = "beta",
    se_col = "se",
    eaf_col = "eaf",
    effect_allele_col ="effect_allele",
    other_allele_col = "other_allele",
    pval_col = "pval",
    samplesize_col= "samplesize",
    chr_col='chr',
    pos_col='pos'
  )
#harmonization
har_dat <- harmonise_data(exposure_dat, outcome_dat, action=2)
#steiger filtering
stei_dat <- steiger_filtering(har_dat)
#MR results
mr_results <- mr(har_dat)
mr_heterogeneity(har_dat)
mr_pleiotropy_test(har_dat)

#rerun Step 2 after replacing i from 2 to nrow(outcomelist)

In [ ]:
##MR analysis for potential mediators


## Step 1: MR analysis testing causal effects of GLP1R bioactivity on BMI/sex hormone
data<-read.csv("multi-omic_qtl_instruments.csv")
#exposure data
data <- data.frame(SNP = data$SNP,
                   chr = data$chr,
                   pos = data$pos,
                   beta = as.numeric(data$beta),
                   se = as.numeric(data$se),
                   effect_allele = data$effect_allele,
                   other_allele = data$other_allele,
                   eaf=data$effect_allele_freq,
                   pval = as.numeric(data$pval),
                   Phenotype = data$Phenotype,
                   samplesize = data$samplesize,
                   info = paste(data$Study,data$cis_trans,sep="_"))
data <- format_data(data, type="exposure", phenotype_col = "Phenotype",chr_col = "chr",pos_col = "pos",samplesize_col = "samplesize",info_col = "info")
#ld clumping
data <-clump_data(data,clump_kb=10000,clump_r2=0.001,clump_p1=1,clump_p2=1,pop="EUR")

# F statistics>10
exposure_dat <- cbind(data, fstatistics = 1)
for (s in 1:nrow(exposure_dat)){
  beta <- exposure_dat[s, "beta.exposure"]
  n <- exposure_dat[s, "samplesize.exposure"]
  eaf <- exposure_dat[s, "eaf.exposure"]

  # Calculate R2 using the formula: R2 = beta^2 * 2 * MAF * (1-MAF)
  r2 <- beta^2 * 2 * eaf * (1 - eaf)

  # Calculate F-statistic
  exposure_dat[s, "fstatistics"] <- (r2 * (n - 2)) / (1 - r2)
}

print(min(exposure_dat$fstatistics))
print(max(exposure_dat$fstatistics))
exposure_dat <- exposure_dat[exposure_dat$fstatistics > 10, ]

#outcome data
outcomelist<-read.csv("outcomelist_mediators.csv")
i<-1
out_data<- vroom(outcomelist[i,"filename"])
outcome_dat<-format_data(
  out_data,
  type = "outcome",
  snp_col = "SNP",
  beta_col = "beta",
  se_col = "se",
  eaf_col = "eaf",
  effect_allele_col ="effect_allele",
  other_allele_col = "other_allele",
  pval_col = "pval",
  samplesize_col= "samplesize",
  chr_col='chr',
  pos_col='pos'
)
#harmonization
har_dat <- harmonise_data(exposure_dat, outcome_dat, action=2)
#steiger filtering
stei_dat <- steiger_filtering(har_dat)
#MR results
mr_results <- mr(har_dat)
mr_heterogeneity(har_dat)
mr_pleiotropy_test(har_dat)
#run the above codes with i from 1 to 7
#remove the instruments of GLP1R bioactivity proxied by pQTLs when the mediator is protein hormone
#save the above results


## Step 2: MR analysis testing causal effects of  BMI/sex hormone  on subtypes of EOC
exposurelist<-read.csv("outcomelist_mediators.csv")
i<-1
exp_data<- vroom(outcomelist[i,"filename"])
exposure_dat<-format_data(
  exp_data,
  type = "exposure",
  snp_col = "SNP",
  beta_col = "beta",
  se_col = "se",
  eaf_col = "eaf",
  effect_allele_col ="effect_allele",
  other_allele_col = "other_allele",
  pval_col = "pval",
  samplesize_col= "samplesize",
  chr_col='chr',
  pos_col='pos'
)
outcomelist<-c("ieu-a-1123","ieu-a-1124","ieu-a-1125","ieu-a-1121","ieu-a-1122")
har_all<data.frame()
mr_all<-data.frame()
stei_all<-data.frame()

for (m in c(1:5)) {
  outcome_dat<- extract_outcome_data(
    snps = exposure_dat$SNP,
    outcomes = outcomelist[m])
  har_dat <- harmonise_data(
    exposure_dat,
    outcome_dat
  )
  #steiger filtering
  stei_dat <- steiger_filtering(har_dat)
  #MR results
  mr_dat <- mr(har_dat)
  het<-mr_heterogeneity(har_dat)
  print(het)
  ple<-mr_pleiotropy_test(har_dat)
  print(ple)

  har_all<-rbind(har_all,har_dat)
  mr_all<-rbind(mr_all,mr_dat)
  stei_all<-rbind(stei_all,stei_dat)

}

#run the above codes with i from 1 to 7
#save the above results
#count the weight of mediators in the association of GLP1R and subtypes of EOC

In [ ]:
##LD_check
library(TwoSampleMR)
library(MendelianRandomization)
library(ieugwasr)
library(openxlsx)

input <- read_excel("LD_check_input.xlsx",sheet=1)
input$SNP
outcome <- read.table("outcome.txt",sep="\t",header=FALSE)

i<-1

for (i in 1:nrow(input)){
  print(i)

  c <- NULL
  d <- input[i,]

  rsid <- input$SNP[i]
  chr<-input$CHR[i]
  p1<-input$Lower[i]
  p2<-input$Upper[i]
  data<-outcome[outcome$chr==chr &outcome$position>p1 & outcome$position<p2,]

  try(data <- data[order(data$p_value),])
  try(data <- data[data$p_value<1E-3,])
  try(if(nrow(data)>=500){data <- data[1:499,] })

  if (nrow(data)!=0){
    rsid <- as.character(unlist(d[1,6]))
    snp <- append(as.character(unlist(data$snp)), rsid)

    a <- NULL
    attempts <- 0
    while(attempts<=10){
      try(a <- ld_matrix(snp))
      if(is.null(a)){
        attempts<-attempts+1}
      else{
        break
      }
    }

    if(is.null(nrow(a))==TRUE){
      c <- as.data.frame(cbind(as.character(unlist(d[1,6])), as.character(unlist(d[1,1])),rsid, 1 ) )
    } else {
      col.index <- which(grepl(rsid,colnames(a)))
      b <- (a[,col.index])^2
      b <- b[order(b)]
      b <- b[(length(b)-1)]
      c <- as.data.frame(cbind(as.character(unlist(d[1,6])),as.character(unlist(d[1,1])), names(b), as.character(unlist(b))) )
    }

  } else {
    c <- as.data.frame(cbind(as.character(unlist(d[1,6])), as.character(unlist(d[1,1])),"NA","NA" ) )
  }

  result_file <- paste0("./ld-check/",as.character(unlist(d[1,6])),".",as.character(unlist(d[1,1])),"_ldcheck")
  if (exists("c")==TRUE){ write.table(c,file=result_file,sep="\t",col.names=F,row.names=F,quote=F)}

}

In [ ]:
##colocalization
rm(list=ls())
gc()
library(TwoSampleMR)
library(coloc)
library(data.table)

#input full summary data of ENOC
data<-fread("chr6_ENOC.txt",header = T)

#step1 eQTL
#rs880347

gwas1 <- fread("eqtl_pancreas.txt",header = T)
gwas1 <- gwas1[gwas1$chr == "chr6",]
gwas1 <- gwas1[gwas1$pos> posx-500000   &  gwas1$pos < posx+500000,]
gwas1$samplesize<-round(gwas1$mac/gwas1$maf/2)
gwas1<-data.frame(gwas1$snp,
                  gwas1$pos,
                  gwas1$beta,
                  gwas1$se,
                  gwas1$eaf,
                  gwas1$p,
                  gwas1$ref,
                  gwas1$alt,
                  gwas1$phenotype_id,
                  gwas1$samplesize
)
colnames(gwas1)<-c("SNP","pos","beta","se","eaf","pval","other_allele","effect_allele","gene_id","N")
gwas2<-data
gwas2<- gwas2[gwas2$Position > posy-500000  &  gwas2$Position < posy+500000,]

colnames(gwas2)<-c("SNP","pos","beta","se","eaf","pval","other_allele","effect_allele","N")
dat_merge <- merge(gwas1,gwas2,by=c("SNP"),suffixes = c("_gwas1","_gwas2"))

dat_merge$label_a1 <- ifelse(dat_merge$effect_allele_gwas1 == dat_merge$effect_allele_gwas2,"yes","no")
dat_merge$label_a1a2 <- ifelse(dat_merge$effect_allele_gwas1 == dat_merge$other_allele_gwas2,"yes","no")
dat_merge$label_a2a1 <- ifelse(dat_merge$other_allele_gwas1 == dat_merge$effect_allele_gwas2,"yes","no")
dat_merge$effect_allele_gwas2 <- ifelse( dat_merge$label_a1a2 == "yes" & dat_merge$label_a2a1 == "yes",
                                          dat_merge$effect_allele_gwas1,
                                          dat_merge$effect_allele_gwas2 )
dat_merge$other_allele_gwas2 <- ifelse( dat_merge$label_a1a2 == "yes" & dat_merge$label_a2a1 == "yes",
                                         dat_merge$other_allele_gwas1,
                                         dat_merge$other_allele_gwas2 )
dat_merge$beta_gwas2 <-ifelse( dat_merge$label_a1a2 == "yes" & dat_merge$label_a2a1 == "yes",
                                -dat_merge$beta_gwas2 ,
                                dat_merge$beta_gwas2)
dat_merge[,c("label_a1","label_a2","label_a1a2","label_a2a1")] <- NULL

dat_merge$MAF_gwas1   <- ifelse(dat_merge$eaf_gwas1   < 0.5,dat_merge$eaf_gwas1,1-dat_merge$eaf_gwas1)
dat_merge$MAF_gwas1[is.na(dat_merge$MAF_gwas1)] <- 0.5
dat_merge$MAF_gwas2 <- ifelse(dat_merge$eaf_gwas2 < 0.5,dat_merge$eaf_gwas2,1-dat_merge$eaf_gwas2)
dat_merge$MAF_gwas2[is.na(dat_merge$MAF_gwas2)] <- 0.5
dat_merge <- dat_merge[dat_merge$MAF_gwas2 != 0, ]


gwas1_form <- list(snp = dat_merge$SNP,
                    position = dat_merge$pos_gwas2,
                    beta = dat_merge$beta_gwas1,
                    varbeta = dat_merge$se_gwas1 ^ 2,
                    MAF = dat_merge$MAF_gwas1,
                    type = "quant",
                    N = dat_merge$N_gwas1)
gwas2_form<- list(snp = dat_merge$SNP,
                   position =dat_merge$pos_gwas2,
                   beta = dat_merge$beta_gwas2,
                   varbeta = dat_merge$se_gwas2^2,
                   MAF=dat_merge$MAF_gwas2,
                   type = "quant",
                   N = dat_merge$N_gwas2)
check_dataset(gwas1_form)
check_dataset(gwas2_form)
my.res <- coloc.abf(dataset1 =gwas1_form ,dataset2 = gwas2_form)

posterior_results <- data.frame(
  nsnps = my.res$summary["nsnps"],
  H0 = my.res$summary["PP.H0.abf"],
  H1 = my.res$summary["PP.H1.abf"],
  H2 = my.res$summary["PP.H2.abf"],
  H3 = my.res$summary["PP.H3.abf"],
  H4 = my.res$summary["PP.H4.abf"]
)
write.csv(posterior_results, file = "coloc_results_eQTL.csv", row.names = FALSE)




#step2 sQTL
#rs10305439
gwas1 <- fread("sqtl_pancreas.txt",header = T)
gwas1 <- gwas1[gwas1$chr == "chr6",]
gwas1 <- gwas1[gwas1$pos> posx-500000   &  gwas1$po < posx+500000,]
gwas1$samplesize<-round(gwas1$mac/gwas1$maf/2)
gwas1<-data.frame(gwas1$snp,
                  gwas1$pos,
                  gwas1$beta,
                  gwas1$se,
                  gwas1$eaf,
                  gwas1$p,
                  gwas1$ref,
                  gwas1$alt,
                  gwas1$phenotype_id,
                  gwas1$samplesize
)
colnames(gwas1)<-c("SNP","pos","beta","se","eaf","pval","other_allele","effect_allele","gene_id","N")
gwas2<-data
gwas2 <- gwas2[gwas2$Chromosome == 6,]
gwas2<- gwas2[gwas2$Position > posy-500000  &  gwas2$Position < posy+500000,]

colnames(gwas2)<-c("SNP","pos","beta","se","eaf","pval","other_allele","effect_allele")
dat_merge <- merge(gwas1,gwas2,by=c("SNP"),suffixes = c("_gwas1","_gwas2"))


dat_merge$label_a1 <- ifelse(dat_merge$effect_allele_gwas1 == dat_merge$effect_allele_gwas2,"yes","no")
dat_merge$label_a1a2 <- ifelse(dat_merge$effect_allele_gwas1 == dat_merge$other_allele_gwas2,"yes","no")
dat_merge$label_a2a1 <- ifelse(dat_merge$other_allele_gwas1 == dat_merge$effect_allele_gwas2,"yes","no")
dat_merge$effect_allele_gwas2 <- ifelse( dat_merge$label_a1a2 == "yes" & dat_merge$label_a2a1 == "yes",
                                         dat_merge$effect_allele_gwas1,
                                         dat_merge$effect_allele_gwas2 )
dat_merge$other_allele_gwas2 <- ifelse( dat_merge$label_a1a2 == "yes" & dat_merge$label_a2a1 == "yes",
                                        dat_merge$other_allele_gwas1,
                                        dat_merge$other_allele_gwas2 )
dat_merge$beta_gwas2 <-ifelse( dat_merge$label_a1a2 == "yes" & dat_merge$label_a2a1 == "yes",
                               -dat_merge$beta_gwas2 ,
                               dat_merge$beta_gwas2)
dat_merge[,c("label_a1","label_a2","label_a1a2","label_a2a1")] <- NULL

dat_merge$MAF_gwas1   <- ifelse(dat_merge$eaf_gwas1   < 0.5,dat_merge$eaf_gwas1,1-dat_merge$eaf_gwas1)
dat_merge$MAF_gwas1[is.na(dat_merge$MAF_gwas1)] <- 0.5
dat_merge$MAF_gwas2 <- ifelse(dat_merge$eaf_gwas2 < 0.5,dat_merge$eaf_gwas2,1-dat_merge$eaf_gwas2)
dat_merge$MAF_gwas2[is.na(dat_merge$MAF_gwas2)] <- 0.5
dat_merge <- dat_merge[dat_merge$MAF_gwas2 != 0, ]

gwas1_form <- list(snp = dat_merge$SNP,
                   position = dat_merge$pos_gwas2,
                   beta = dat_merge$beta_gwas1,
                   varbeta = dat_merge$se_gwas1 ^ 2,
                   MAF = dat_merge$MAF_gwas1,
                   type = "quant",
                   N = dat_merge$N_gwas1)
gwas2_form<- list(snp = dat_merge$SNP,
                  position =dat_merge$pos_gwas2,
                  beta = dat_merge$beta_gwas2,
                  varbeta = dat_merge$se_gwas2^2,
                  MAF=dat_merge$MAF_gwas2,
                  type = "quant",
                  N = dat_merge$N_gwas2)
check_dataset(gwas1_form)
check_dataset(gwas2_form)
my.res <- coloc.abf(dataset1 =gwas1_form ,dataset2 = gwas2_form)

posterior_results <- data.frame(
  nsnps = my.res$summary["nsnps"],
  H0 = my.res$summary["PP.H0.abf"],
  H1 = my.res$summary["PP.H1.abf"],
  H2 = my.res$summary["PP.H2.abf"],
  H3 = my.res$summary["PP.H3.abf"],
  H4 = my.res$summary["PP.H4.abf"]
)

write.csv(posterior_results, file = "coloc_results_sQTL.csv", row.names = FALSE)